# Setup

**Information about the different models which might be useful when writing the paper**
C-RADIOv2 models are available in multiple sizes: Base (90M parameters). Huge (653M parameters). C-RADIOv2 was trained for 1M steps (400k more steps than v1), using inverse frequency sampling for data balancing, and PHI Standardization for teacher distribution balancing. 

Model Architecture
Architecture Type: Neural Network
Network Architecture: Vision Transformer

c-radio-h	ViT-H/16-CPE https://github.com/NVlabs/RADIO

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import string
import os
from pathlib import Path

In [ ]:
DF_1024000 = pd.read_csv("../results/results_exp_a_500_sharding_batch4_workers8_dataparallel_memory1024000.csv")  # the main memory size used throughout experiments
DF_640000 = pd.read_csv("../results/results_exp_a_500_sharding_batch4_workers8_dataparallel_memory640000.csv")  # only used in Experiment C
DF_320000 = pd.read_csv("../results/results_exp_a_500_sharding_batch4_workers8_dataparallel_memory320000.csv")  # only used in Experiment C


In [ ]:
MODELS = {
    "clip-vit-base-patch16": "CLIP",
    "dino_vitb16": "DINO",
    "dinov2_vitb14": "DINOv2",
    "dinov3-vitb16-pretrain-lvd1689m": "DINOv3",
    "C-RADIOv2-B": "C-RADIOv2",
    "siglip2-base-patch16-512": "SigLIP2",
    "tips-b14": "TIPS",
}
MODELS_FULL = {
    "clip-vit-base-patch16": "CLIP ViT-B/16",
    "dino_vitb16": "DINO ViT-B/16",
    "dinov2_vitb14": "DINOv2 ViT-B/14",
    "dinov3-vitb16-pretrain-lvd1689m": "DINOv3 ViT-B/16",
    "C-RADIOv2-B": "C-RADIOv2 ViT-B/16-CPE",
    "siglip2-base-patch16-512": "SigLIP2 B/16-512",
    "tips-b14": "TIPS ViT-B/14-HR",
}
PREFIXES = {
    "clip-vit-base-patch16": "clip",
    "dino_vitb16": "dino",
    "dinov2_vitb14": "dinov2",
    "dinov3-vitb16-pretrain-lvd1689m": "dinov3",
    "C-RADIOv2-B": "radio",
    "siglip2-base-patch16-512": "siglip2",
    "tips-b14":"tips",
}
COLORS = {
    "clip-vit-base-patch16": "#1f77b4",
    "dino_vitb16": "#ff7f0e",
    "dinov2_vitb14": "#2ca02c",
    "dinov3-vitb16-pretrain-lvd1689m": "#17becf",
    "C-RADIOv2-B": "#d62728",
    "siglip2-base-patch16-512": "#9467bd",
    "tips-b14": "#8c564b",
}
TRAIN_BINS_DIFFICULTY = {
    "0_30_60_90": "easy",
    "0_45_90": "medium",
    "0_90": "hard",
    "0": "extreme"
}
XTICKS = [0, 15, 30, 45, 60, 75, 90]
CLASS_TO_INDEX = {
    0: 0,
    7: 1,
    8: 2,
    19: 3,
    46: 4,
    57: 5,
    60: 6,
    70: 7,
    99: 8,
    100: 9,
    113: 10,
    125: 11,
    126: 12,
    152: 13,
    166: 14,
    196: 15,
}
INDEX_TO_CLASS_ID = {idx: cid for cid, idx in CLASS_TO_INDEX.items()}  # reverse mapping: class_idx -> class_id
CLASS_TO_NAME = {
    0: "background",
    7: "stove",
    8: "sofa",
    19: "microwave",
    46: "bed",
    57: "toy cat",
    60: "toy cow",
    70: "toy dragon",
    99: "coat rack",
    100: "guitar stand",
    113: "ceiling lamp",
    125: "toilet",
    126: "sink",
    152: "strings",
    166: "broccoli",
    196: "durian"
}
BP_THRESHOLD = -0.1
DIFFS = ["easy", "medium", "hard", "extreme"]
JAC_COLS = [f"jac{i}" for i in range(1, 16)]



# Experiments A and C

In [ ]:
DF_1024000.head(3)

## Plots

In [ ]:
def plot_exp_a(classes: None | list[int], include_bg: bool, extratitle: str, save_path: str | None = None):
    assert isinstance(classes, (list, type(None))), "classes must be a list or None"

    if classes is None:
        classes = list(CLASS_TO_INDEX.keys())
    else:
        assert all(c in CLASS_TO_INDEX for c in classes), "classes must be a subset of CLASS_TO_INDEX keys"

    if not include_bg:
        classes = [c for c in classes if c != 0]

    class_indices = [CLASS_TO_INDEX[c] for c in classes]
    jac_cols = [f"jac{c}" for c in class_indices]

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), dpi=200)
    axes = axes.flatten()

    # Which x-positions to hide (set to 0), per subplot index i
    bar_zero_idx = {
        0: [0, 2, 4, 6],
        1: [0, 3, 6],
        2: [0, 6],
        3: [0],
    }
    scatter_zero_idx = {
        0: [1, 3, 5],
        1: [1, 2, 4, 5],
        2: [1, 2, 3, 4, 5],
        3: [1, 2, 3, 4, 5, 6],
    }

    bar_width = 0.1
    bin_vals = sorted(set(DF_1024000["val_bin"]))
    x_pos = np.arange(len(bin_vals))
    total_models = len(MODELS)

    for i, (train_bins, difficulty) in enumerate(TRAIN_BINS_DIFFICULTY.items()):
        ax = axes[i]

        for model_idx, (model, model_label) in enumerate(MODELS.items()):
            df = DF_1024000[(DF_1024000["model"] == model) & (DF_1024000["train_bins"] == train_bins)].copy()
            if df.empty:
                print(f"No data for model {model} with train_bins {train_bins} and classes {classes}")
                continue

            # Robust alignment to bins (safe even if a bin is missing)
            df = df.set_index("val_bin").reindex(bin_vals).reset_index()

            means_base = df[jac_cols].mean(axis=1).to_numpy()

            # Bar means
            means_bar = means_base.copy()
            means_bar[bar_zero_idx.get(i, [])] = 0

            # Scatter means
            means_scatter = means_base.copy()
            means_scatter[scatter_zero_idx.get(i, [])] = 0

            offset = (model_idx - total_models / 2) * bar_width + bar_width / 2
            x_offset = x_pos + offset

            ax.bar(
                x_offset, means_bar, width=bar_width, label=model_label,
                color=COLORS[model], capsize=4, zorder=2
            )
            ax.scatter(x_offset, means_scatter, marker=".", color=COLORS[model], s=50, zorder=3)

        ax.set_xticks(x_pos)
        ax.set_xticklabels([str(b) for b in bin_vals])
        ax.set_ylim(0, 1)
        ax.set_title(f"({string.ascii_letters[i]}) Train bin(s): {', '.join(train_bins.split('_'))} ({difficulty})")
        ax.set_xlabel("Validation Bin")
        ax.set_ylabel("mIoU")
        ax.grid(True, axis="y", linestyle="--", linewidth=0.5)

    axes[0].legend(loc="lower left", bbox_to_anchor=(0, 0))
    plt.suptitle(f"Experiment A Results — {extratitle}", fontsize=25, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"Saved figure to {save_path}")

    plt.show()
    plt.close()


In [ ]:
# Experiment A (Grouped Bar Histograms)
os.makedirs("../images/figures", exist_ok=True)

# Plot all classes
plot_exp_a(None, False, extratitle="All classes excl. bg", save_path="../images/figures/All_Classes_No_BG.png")
plot_exp_a(None, True, extratitle="All classes", save_path="../images/figures/All_Classes.png")

In [ ]:
# Plot each class separately
for idx, (class_id, class_name) in enumerate(CLASS_TO_NAME.items()):
    save_path = f"../images/figures/{idx}_{class_name.replace(' ', '_')}.png"
    if class_id == 0:
        plot_exp_a([class_id], include_bg=True, extratitle=class_name.capitalize(), save_path=save_path)
    else:
        plot_exp_a([class_id], include_bg=False, extratitle=class_name.capitalize(), save_path=save_path)


## Comparing mIoU per class to the mIoU over all classes

In [ ]:
df = DF_1024000.copy()

# Per-row baseline: average mIoU across all classes
# Used to measure whether a specific class over/underperforms
class_cols = [f"jac{idx}" for idx in CLASS_TO_INDEX.values()]
df["all_classes_mIoU"] = df[class_cols].mean(axis=1)

# Map training-bin configurations to semantic difficulty levels
df["difficulty"] = df["train_bins"].map(TRAIN_BINS_DIFFICULTY)

# Convert per-class columns (jac0..jacN) into long format
# Long format enables clean grouping by class, model, and difficulty
long = df.melt(
    id_vars=["model", "difficulty", "train_bins", "val_bin", "all_classes_mIoU"],
    value_vars=class_cols,
    var_name="jac_col",
    value_name="class_mIoU",
)

# Recover class indices from column names (jac<i>)
long["class_idx"] = long["jac_col"].str.replace("jac", "", regex=False).astype(int)

# Map numeric indices back to semantic class IDs and names
long["class_id"] = long["class_idx"].map(INDEX_TO_CLASS_ID)
long["class_name"] = long["class_id"].map(CLASS_TO_NAME)
long["class_full"] = long["class_id"].astype(str) + " (" + long["class_name"] + ")"

# Difference from the row-wise baseline
# Normalizes class performance within each evaluation setting
long["difference"] = long["class_mIoU"] - long["all_classes_mIoU"]

table = long.drop(columns=["jac_col"])

# Identify classes that consistently under- or overperform
class_diff_mean = (
    table.groupby("class_full")["difference"]
    .mean()
    .sort_values()
)

# Same analysis broken down by model and difficulty
diff_by_model_difficulty = (
    table.groupby(["model", "difficulty", "class_full"])["difference"]
    .mean()
)

# Parse training bins to detect whether the validation bin was seen during training
train_bins_list = (
    table["train_bins"]
    .astype(str)
    .str.split("_")
    .apply(lambda xs: [int(x) for x in xs])
)

# Flag generalization cases (unseen viewpoint bins)
table["is_seen_bin"] = [
    vb in tb for vb, tb in zip(table["val_bin"], train_bins_list)
]

unseen = table[~table["is_seen_bin"]].copy()

# Class-wise relative performance on unseen bins only
unseen_diff = (
    unseen.groupby(["model", "difficulty", "class_full"])["difference"]
    .mean()
)

# Absolute class mIoU on unseen bins
unseen_miou = (
    unseen.groupby(["model", "difficulty", "class_full"])["class_mIoU"]
    .mean()
)

# High-level summaries for tables or figures
pivot_unseen_miou = pd.pivot_table(
    unseen,
    values="class_mIoU",
    index="difficulty",
    columns="model",
    aggfunc="mean",
)

pivot_unseen_diff = pd.pivot_table(
    unseen,
    values="difference",
    index="difficulty",
    columns="model",
    aggfunc="mean",
)



In [ ]:
pivot_unseen_miou

In [ ]:
pivot_unseen_diff

## Job Statistics

In [ ]:
job_id = str(DF_1024000['job_id'].iloc[0])
model = DF_1024000['model'].iloc[0]
prefix = PREFIXES.get(model)
if prefix is None:
    raise ValueError(f"Unknown model name: {model}")

log_path = Path(f"../logs/exp_a_b/{prefix}.job_{job_id}.log")

# Read last 15 lines after JOB STATISTICS
def tail_log_after_statistics(path, num_lines=15):
    if not path.exists():
        return f"Log file not found: {path}"
    
    with open(path, 'r') as f:
        lines = f.readlines()
        try:
            start_idx = next(i for i, line in enumerate(lines) if 'JOB STATISTICS' in line)
            return ''.join(lines[start_idx:start_idx + num_lines])
        except StopIteration:
            return "JOB STATISTICS not found in log file."

# Output result
log_tail = tail_log_after_statistics(log_path)
print(log_tail)


In [ ]:
# # Get all unique (job_id, model) pairs
# unique_jobs = DF_1024000.copy()
# unique_jobs = unique_jobs[['job_id', 'model']].drop_duplicates()

# # Function to extract JOB STATISTICS section
# def print_job_statistics(job_id, model):
#     prefix = PREFIXES.get(model)
#     if not prefix:
#         print(f"Skipping unknown model: {model}")
#         return
    
#     log_path = Path(f"../logs/exp_a_b/{prefix}.job_{job_id}.log")
#     print(f"\n=== JOB STATISTICS for job_id: {job_id}, model: {model} ===")
    
#     if not log_path.exists():
#         print(f"Log file not found: {log_path}")
#         return

#     with open(log_path, 'r') as f:
#         lines = f.readlines()
#         try:
#             start_idx = next(i for i, line in enumerate(lines) if 'JOB STATISTICS' in line)
#             for line in lines[start_idx:]:
#                 print(line, end='')  # already includes newline
#         except StopIteration:
#             print("'JOB STATISTICS' section not found in log.")

# # Loop and print
# print(f"The following JOB STATISTICS are for memory: {MEMORY}")
# for _, row in unique_jobs.iterrows():
#     job_id = str(row['job_id'])
#     model = row['model']
#     print_job_statistics(job_id, model)


## Tables

In [ ]:
def build_memory_segment_numeric(df_in: pd.DataFrame) -> pd.DataFrame:
    """
    Returns table segment containing:
      model, easy_mean, easy_std, medium_mean, medium_std, ...
    """
    df = df_in.copy()

    df["train_bins"] = df["train_bins"].apply(lambda x: list(map(int, str(x).split("_"))))  # parse train_bins "0_1_2" -> [0,1,2]
    df["val_in_trained"] = df.apply(lambda row: row["val_bin"] in row["train_bins"], axis=1)  # if val_bin is in train_bins
    df["difficulty"] = df["train_bins"].apply(lambda x: TRAIN_BINS_DIFFICULTY["_".join(map(str, x))])

    df = df[~df["val_in_trained"]].copy()  # skip the validation bins that were seen during training

    df["jac_mean"] = df[JAC_COLS].mean(axis=1)
    df["jac_std"] = df[JAC_COLS].std(axis=1)

    rows = []
    for model in MODELS_FULL.keys():
        mdf = df[df["model"] == model]

        means = mdf.groupby("difficulty")["jac_mean"].mean()
        stds = mdf.groupby("difficulty")["jac_std"].mean()

        row = {"model": MODELS_FULL[model]}
        for d in DIFFS:
            row[f"{d}_mean"] = float(means.get(d, 0.0))
            row[f"{d}_std"] = float(stds.get(d, 0.0))
        rows.append(row)

    cols = ["model"] + [f"{d}_{s}" for d in DIFFS for s in ["mean", "std"]]
    return pd.DataFrame(rows, columns=cols)


def format_segment(seg_num: pd.DataFrame) -> pd.DataFrame:
    """
    Takes numeric table segment from build_memory_segment_numeric()
    Returns the formatted table segment containing:
      model, easy, medium, hard, extreme
    """
    out = pd.DataFrame({"model": seg_num["model"]})
    for d in DIFFS:
        out[d] = seg_num.apply(lambda r: f"{r[f'{d}_mean']:.3f} $\pm$ {r[f'{d}_std']:.3f}", axis=1)
    return out[["model"] + DIFFS]

seg320_num = build_memory_segment_numeric(DF_320000)
seg640_num = build_memory_segment_numeric(DF_640000)
seg1024_num = build_memory_segment_numeric(DF_1024000)

seg_320k = format_segment(seg320_num)
seg_640k = format_segment(seg640_num)
seg_1024k = format_segment(seg1024_num)


In [ ]:
def segment_to_latex(df_seg: pd.DataFrame) -> str:
    return df_seg.to_latex(
        index=False,
        escape=False,
        column_format="lcccc",
        header=["Model", "Easy", "Medium", "Hard", "Extreme"],
    )

latex_320k = segment_to_latex(seg_320k)
latex_640k = segment_to_latex(seg_640k)
latex_1024k = segment_to_latex(seg_1024k)

print("--- 320k ---")
print(latex_320k)
print("--- 640k ---")
print(latex_640k)
print("--- 1024k ---")
print(latex_1024k)


In [ ]:
def segment_diff_means(seg_a_num: pd.DataFrame, seg_b_num: pd.DataFrame) -> pd.DataFrame:
    """
    Returns the differences of mIoU per model between segments:
      model, easy, medium, hard, extreme
    """
    a = seg_a_num.set_index("model")
    b = seg_b_num.set_index("model")

    diff = pd.DataFrame(index=a.index)
    for d in DIFFS:
        diff[d] = (b[f"{d}_mean"] - a[f"{d}_mean"])
    return diff.reset_index()

diff_320_640 = segment_diff_means(seg320_num, seg640_num)  # 640 - 320
diff_640_1024 = segment_diff_means(seg640_num, seg1024_num)  # 1024 - 640
diff_320_1024 = segment_diff_means(seg320_num, seg1024_num)  # 1024 - 320


In [ ]:
# def format_diff_segment(diff_df: pd.DataFrame) -> pd.DataFrame:
#     out = diff_df.copy()
#     out["average"] = out[DIFFS].mean(axis=1)

#     # round + stringify
#     for c in DIFFS + ["average"]:
#         out[c] = out[c].round(3).map(lambda v: f"{v:.3f}")

#     return out[["model"] + DIFFS + ["average"]]

def format_diff_segment(diff_df: pd.DataFrame) -> pd.DataFrame:
    out = diff_df.copy()

    # Per-model average (row-wise)
    out["average"] = out[DIFFS].mean(axis=1)

    # Per-task average (column-wise)
    avg_task = out[DIFFS].mean(axis=0)
    avg_task["average"] = avg_task.mean()
    avg_task["model"] = "Average per task"

    # Append as last row
    out = pd.concat(
        [out, avg_task.to_frame().T],
        ignore_index=True
    )

    # Ensure numeric dtype before rounding
    for c in DIFFS + ["average"]:
        out[c] = pd.to_numeric(out[c], errors="raise")

    # Round and stringify for display
    for c in DIFFS + ["average"]:
        out[c] = out[c].round(3).map(lambda v: f"{v:.3f}")

    return out[["model"] + DIFFS + ["average"]]

gain_320_640 = format_diff_segment(diff_320_640)
gain_640_1024 = format_diff_segment(diff_640_1024)
gain_320_1024 = format_diff_segment(diff_320_1024)

print("320k to 640k")
display(gain_320_640)
print("640k to 1,024k")
display(gain_640_1024)
print("320k to 1,024k")
display(gain_320_1024)


In [ ]:
def latex_table_body(df: pd.DataFrame) -> str:
    latex = df.to_latex(
        index=False,
        escape=False,
        header=False,
        column_format="lccccc",
    )
    lines = latex.splitlines()
    start = next(i for i, l in enumerate(lines) if "\\toprule" in l) + 1
    end = next(i for i, l in enumerate(lines) if "\\bottomrule" in l)
    return "\n".join(lines[start:end])

print(latex_table_body(gain_320_640))
print()
print(latex_table_body(gain_640_1024))
print()
print(latex_table_body(gain_320_1024))


# Experiment B

In [ ]:
DF_EXTREME = DF_1024000[DF_1024000["train_bins"] == "0"].copy()  # extreme difficulty
DF_EXTREME.head(3)

## Plots

In [ ]:
def plot_exp_b(normalize=False, save_path: str | None = None):
    fig, ax = plt.subplots(figsize=(6, 4), dpi=200)

    if normalize:
        # plt.scatter(0, 1, color="black", marker="x", zorder=5, label="Baseline performance")
        plt.scatter(0, 1, color="black", marker="x", zorder=5, label="Baseline")
        plot_title = "Experiment B results (normalized to 0-bin performance)"
        y_label = "Normalized mIoU"
    else:
        plot_title = "Experiment B results"
        y_label = "mIoU"

    for model in MODELS.keys():
        df = DF_EXTREME[DF_EXTREME["model"] == model].copy()
        df["jac_mean"] = df[[f"jac{i}" for i in range(1, 16)]].mean(axis=1)
        df["jac_std"] = df[[f"jac{i}" for i in range(1, 16)]].std(axis=1)

        if not normalize:
            # plot the 0-bin performance
            df0 = df[df["val_bin"] == 0]
            x_values = df0["val_bin"]
            y_values = df0["jac_mean"]
            plt.scatter(x_values, y_values, color=COLORS[model], marker="x", s=100, zorder=5)

        # plot the other validation bins [15, ..., 90]
        if normalize:
            # get bin=0 performance
            df0 = df[df["val_bin"] == 0]
            model_0_score = df0["jac_mean"].values[0]

            df1590 = df[df["val_bin"] != 0].copy()
            df1590["jac_norm"] = df1590["jac_mean"] / model_0_score
            x_values = df1590["val_bin"]
            y_values = df1590["jac_norm"]
        else:
            df1590 = df[df["val_bin"] != 0]
            x_values = df1590["val_bin"]
            y_values = df1590["jac_mean"].values

        plt.plot(x_values, y_values, label=MODELS[model], color=COLORS[model])

    plt.xlabel("Bin")
    plt.ylabel(y_label)
    plt.title(plot_title)
    plt.xticks(XTICKS)
    plt.legend(fontsize='small')
    plt.grid(True)
    plt.ylim(0, 1.1)
    plt.tight_layout()

    # Save figure
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved figure to {save_path}")
    
    plt.show()


In [ ]:
plot_exp_b(normalize=True, save_path="../images/expb_norm.png")
plot_exp_b(normalize=False, save_path="../images/expb_raw.png")

## Table

We normalize mIoU by the 0 degree bin and compute the change between consecutive viewpoint bins. The breaking point is the first bin where this normalized drop exceeds a fixed threshold. If no bin satisfies this condition, the model is treated as having no breaking point.

In [ ]:
df_break = DF_EXTREME.copy()  # Breaking point dataframe

jac_cols = [f"jac{i}" for i in range(1, 16)]
DF_EXTREME["jac_mean"] = DF_EXTREME[jac_cols].mean(axis=1)
DF_EXTREME["jac_std"]  = DF_EXTREME[jac_cols].std(axis=1)
final_df = None

for model in MODELS.keys():
     # Get the extreme difficulty for the current model
    df = df_break[df_break["model"] == model].copy()
    
    # Ensure consecutive rows correspond to increasing viewpoint angle
    df = df.sort_values("val_bin").reset_index(drop=True)

    # Baseline is validating on the training bin (0 degrees)
    base = df.loc[df["val_bin"] == 0, "jac_mean"]
    if base.empty:
        raise ValueError(f"Missing val_bin == 0 for model={model}")
    model_extreme_score = float(base.mean())

    # Divide the mean by the 0-bin baseline to get normalized mIoU
    df["jac_norm"] = df["jac_mean"] / model_extreme_score

    # Find drop between consecutive bins
    df["delta"] = df["jac_norm"].diff()
    
    # remove reference bin (no valid delta at 0 degrees)
    # df = df[df["val_bin"] != 0].copy()
    df = df[~df["val_bin"].isin([0, 15])].copy()

    # Add model to the dataframe)
    final_df = df if final_df is None else pd.concat(
        [final_df, df], ignore_index=True
    )

final_df

In [ ]:
# Summary dataframe of breakpoints (one row per model)
rows = []
for key, pretty in MODELS_FULL.items():
    g = final_df[final_df["model"] == key].sort_values("val_bin")
    if g.empty:
        continue

    bp = g[g["delta"] <= BP_THRESHOLD].head(1)

    rows.append({
        "Model": pretty,
        "Breaking point bin": int(bp["val_bin"].iloc[0]) if len(bp) else "None",
        # "BP drop": bp["delta"].iloc[0] if len(bp) else None,
        # "Max drop bin": int(g.loc[g["delta"].idxmin()]["val_bin"]),
        "Biggest drop": g["delta"].min(),
    })

bp_df = pd.DataFrame(rows)

bp_df


In [ ]:
# Generate LaTeX table
def fmt(x):
    return "--" if pd.isna(x) else rf"\({x:.4f}\)"

latex_df = bp_df.copy()
# latex_df["BP drop"] = latex_df["BP drop"].apply(fmt)
latex_df["Biggest drop"] = latex_df["Biggest drop"].apply(fmt)
latex_df.columns = [rf"\textbf{{{c}}}" for c in latex_df.columns]

latex = latex_df.to_latex(
    index=False,
    escape=False,
    column_format="lccc",
)

print(latex)
